In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession\
                    .builder\
                    .master("spark://spark-master:7077")\
                    .appName("Day_5_Assignment")\
                    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 10:52:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [5]:
orders_schema_struct = StructType(
                                    [
                                        StructField("order_id",IntegerType()),
                                        StructField("customer_id",IntegerType()),
                                        StructField("product_id",IntegerType()),
                                        StructField("price",FloatType()),
                                        StructField("order_date",DateType()),
                                        StructField("order_status",StringType()),
                                        StructField("state",StringType()),
                                        StructField("quantity",IntegerType()),
                                    ]
                                )

df_orders = spark.read.csv('/data/orders_50mb.csv',schema=orders_schema_struct,header=True)

df_orders.show(5)

+--------+-----------+----------+------+----------+------------+-------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|  state|quantity|
+--------+-----------+----------+------+----------+------------+-------+--------+
|       1|     192520|        70|152.33|2007-10-10|      PLACED| Odisha|       2|
|       2|     835421|        34|117.42|2008-12-02|     SHIPPED| Kerala|       3|
|       3|     159165|        13|128.85|2010-04-22|   DELIVERED|Gujarat|       3|
|       4|     403890|        25|195.71|2007-05-30|   CANCELLED|  Bihar|       1|
|       5|     273746|        41|125.08|2003-12-13|     SHIPPED| Odisha|       3|
+--------+-----------+----------+------+----------+------------+-------+--------+
only showing top 5 rows



In [6]:
# create tempview of this df_orders
df_orders.createOrReplaceTempView('sql_orders')

spark.sql(
    '''
    select * from sql_orders limit 5
    '''
).show()

[Stage 1:>                                                          (0 + 1) / 1]

+--------+-----------+----------+------+----------+------------+-------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|  state|quantity|
+--------+-----------+----------+------+----------+------------+-------+--------+
|       1|     192520|        70|152.33|2007-10-10|      PLACED| Odisha|       2|
|       2|     835421|        34|117.42|2008-12-02|     SHIPPED| Kerala|       3|
|       3|     159165|        13|128.85|2010-04-22|   DELIVERED|Gujarat|       3|
|       4|     403890|        25|195.71|2007-05-30|   CANCELLED|  Bihar|       1|
|       5|     273746|        41|125.08|2003-12-13|     SHIPPED| Odisha|       3|
+--------+-----------+----------+------+----------+------------+-------+--------+



1. From the dataset, create a result showing `state` and `order_year` with
total quantity and total revenue, excluding orders with status `CANCELLED`,
and keep only those state–year combinations where total revenue exceeds 550000.


In [6]:
#pratice
df_orders.filter(upper(col('order_status'))!='CANCELLED')\
            .withColumn('order_year',year(col('order_date')))\
            .groupBy(col('state'),col('order_year'))\
                .agg(
                    sum('quantity').alias('total_quantity') , 
                    sum(col('price')*col('quantity')).alias('total_revenue')
                    )\
            .filter(col('total_revenue')>550000).show()

+-----------+----------+--------------+-----------------+
|      state|order_year|total_quantity|    total_revenue|
+-----------+----------+--------------+-----------------+
|      Bihar|      2020|          3653|551408.4500656128|
|Maharashtra|      2006|          3763|564980.0401229858|
|      Assam|      2006|          3721|555672.2194595337|
|     Odisha|      2018|          3686|551947.6004257202|
|      Bihar|      2016|          3770|568748.1598434448|
|Maharashtra|      2015|          3654|551441.4298934937|
|Uttarakhand|      2020|          3735|556238.4301223755|
|      Delhi|      2020|          3702|558023.8393783569|
| Tamil Nadu|      2019|          3716|559179.5512008667|
+-----------+----------+--------------+-----------------+



In [11]:
spark.sql(
        '''
            select state,year(order_date) as year,sum(quantity) as total_quantity,sum(price*quantity) as revenue
            from sql_orders
            where order_status!='CANCELLED'
            group by state,year(order_date)
            having sum(price*quantity)>550000
            order by revenue desc
        '''
        ).show()

[Stage 5:====================================>                  (131 + 4) / 200]

+-----------+----+--------------+-----------------+
|      state|year|total_quantity|          revenue|
+-----------+----+--------------+-----------------+
|      Bihar|2016|          3770|568748.1598434448|
|Maharashtra|2006|          3763|564980.0401229858|
| Tamil Nadu|2019|          3716|559179.5512008667|
|      Delhi|2020|          3702|558023.8393783569|
|Uttarakhand|2020|          3735|556238.4301223755|
|      Assam|2006|          3721|555672.2194595337|
|     Odisha|2018|          3686|551947.6004257202|
|Maharashtra|2015|          3654|551441.4298934937|
|      Bihar|2020|          3653|551408.4500656128|
+-----------+----+--------------+-----------------+



# Question 2

2. Add a column `order_value_category` where orders with total amount greater than 400 are labeled `HIGH_VALUE`, between 250 and 400 as `MEDIUM_VALUE`, and the rest as `LOW_VALUE`, then return the count of orders in each category.

In [87]:
#pratice

df_orders.withColumn('order_total_amount',col('price')*col('quantity'))\
            .withColumn('order_value_category',when(col('order_total_amount')>400,'HIGH_VALUE')\
                        .when(((col('order_total_amount')<400) & (col('order_total_amount')>250)),'MEDIUM_VALUE')\
                        .otherwise('LOW_VALUE'))\
            .groupBy(col('order_value_category'))\
                .agg(count('*').alias('category_counts'))\
            .show()

+--------------------+---------------+
|order_value_category|category_counts|
+--------------------+---------------+
|          HIGH_VALUE|         222239|
|        MEDIUM_VALUE|         361193|
|           LOW_VALUE|         416568|
+--------------------+---------------+



In [19]:
#need subcategory or cte
spark.sql('''
            select order_value_category,count(*) from 
            (
            select *,
                CASE
                    when price*quantity >400 then 'HIGH_VALUE'
                    when ((price*quantity >250) and (price*quantity <400)) then 'MEDIUM_VALUE'
                    else 'LOW_VALUE'
                END as order_value_category
                from sql_orders
            ) a
            group by order_value_category

        ''').show(5)

+--------------------+--------+
|order_value_category|count(1)|
+--------------------+--------+
|          HIGH_VALUE|  222239|
|        MEDIUM_VALUE|  361193|
|           LOW_VALUE|  416568|
+--------------------+--------+



In [21]:
#cte
spark.sql('''
            with order_cte
            as
            (select *,
                            CASE
                                when price*quantity >400 then 'HIGH_VALUE'
                                when ((price*quantity >250) and (price*quantity <400)) then 'MEDIUM_VALUE'
                                else 'LOW_VALUE'
                            END as order_value_category
                            from sql_orders
            )
            select order_value_category,count(*)
            from order_cte
            group by order_value_category
''').show()


+--------------------+--------+
|order_value_category|count(1)|
+--------------------+--------+
|          HIGH_VALUE|  222239|
|        MEDIUM_VALUE|  361193|
|           LOW_VALUE|  416568|
+--------------------+--------+



# IMP Question-3

3. For each `state`, calculate the total number of delivered orders and the total number of orders, and compute the delivery percentage for that state.

In [9]:
#pratice
df_orders.groupBy(col('state'))\
            .agg(
                    count('*').alias('total_no_of_orders'),
                    sum(when(lower(col('order_status'))=='delivered',1).otherwise(0)).alias('total_delivered_orders')
                )\
        .withColumn('%delivery',(col('total_delivered_orders')/col('total_no_of_orders'))*100)\
        .show()


+----------------+------------------+----------------------+------------------+
|           state|total_no_of_orders|total_delivered_orders|         %delivery|
+----------------+------------------+----------------------+------------------+
|       Karnataka|             45555|                 11455|25.145428602787838|
|          Odisha|             45719|                 11459| 25.06397777729172|
|          Kerala|             45691|                 11490| 25.14718434702677|
|      Tamil Nadu|             45638|                 11386|24.948507822428677|
|    Chhattisgarh|             45562|                 11395| 25.00987665159563|
|  Andhra Pradesh|             45650|                 11410|24.994523548740418|
|  Madhya Pradesh|             45254|                 11338|25.054138860653204|
|          Punjab|             45479|                 11333|24.919193473911037|
|             Goa|             45090|                 11184|24.803725881570195|
|Himachal Pradesh|             45686|   

In [15]:
df_orders.groupBy('state')\
            .agg(count('*').alias('total_orders')\
            ,sum(when(lower(col('order_status'))=='delivered' ,1).otherwise(0)).alias('total_delivered')
                ).withColumn('delivery_percentage',(col('total_delivered')/col('total_orders'))*100)\
                .show()

+----------------+------------+---------------+-------------------+
|           state|total_orders|total_delivered|delivery_percentage|
+----------------+------------+---------------+-------------------+
|       Karnataka|       45555|          11455| 25.145428602787838|
|          Odisha|       45719|          11459|  25.06397777729172|
|          Kerala|       45691|          11490|  25.14718434702677|
|      Tamil Nadu|       45638|          11386| 24.948507822428677|
|    Chhattisgarh|       45562|          11395|  25.00987665159563|
|  Andhra Pradesh|       45650|          11410| 24.994523548740418|
|  Madhya Pradesh|       45254|          11338| 25.054138860653204|
|          Punjab|       45479|          11333| 24.919193473911037|
|             Goa|       45090|          11184| 24.803725881570195|
|Himachal Pradesh|       45686|          11434|  25.02736067942039|
|         Haryana|       45500|          11306|  24.84835164835165|
|       Jharkhand|       45490|          11292| 

In [17]:
#spark.sql

spark.sql('''
            with order_cte as
                (
                    select state,count(*) as total_no_of_orders,
                    sum(case
                        when lower(order_status)=='delivered' then 1
                        else 0
                    END )as total_delived_orders
                    from sql_orders
                    group by state
                )
            select *,((total_delived_orders/total_no_of_orders)*100 ) as pct_delivery
            from order_cte
        ''').show()

+----------------+------------------+--------------------+------------------+
|           state|total_no_of_orders|total_delived_orders|      pct_delivery|
+----------------+------------------+--------------------+------------------+
|       Karnataka|             45555|               11455|25.145428602787838|
|          Odisha|             45719|               11459| 25.06397777729172|
|          Kerala|             45691|               11490| 25.14718434702677|
|      Tamil Nadu|             45638|               11386|24.948507822428677|
|    Chhattisgarh|             45562|               11395| 25.00987665159563|
|  Andhra Pradesh|             45650|               11410|24.994523548740418|
|  Madhya Pradesh|             45254|               11338|25.054138860653204|
|          Punjab|             45479|               11333|24.919193473911037|
|             Goa|             45090|               11184|24.803725881570195|
|Himachal Pradesh|             45686|               11434| 25.02

# Question 4

4. Create a DataFrame showing `customer_id`, number of distinct order years, and total spending, and return only customers who have placed orders in at least 5 distinct years.

In [38]:
#pratice

df = df_orders.withColumn("order_year", year(to_date("order_date"))) \
              .withColumn("order_value", col("price") * col("quantity")) \
              .groupBy("customer_id") \
              .agg(
                  countDistinct("order_year").alias("distinct_years"),
                  sum("order_value").alias("total_spending")
              ) \
              .filter(col("distinct_years") >= 5)\
                .sort(col('customer_id').desc(),col('distinct_years').desc())\
                
df.show()

[Stage 155:=================================================>   (188 + 4) / 200]

+-----------+--------------+------------------+
|customer_id|distinct_years|    total_spending|
+-----------+--------------+------------------+
|     999747|             5|1927.0099792480469|
|     999502|             6|2031.1599884033203|
|     998877|             5|1861.1299285888672|
|     998725|             6| 2014.740005493164|
|     998362|             5|1849.7599792480469|
|     998347|             5| 1494.969985961914|
|     997974|             5|1628.3100357055664|
|     997930|             5|1911.6500091552734|
|     997612|             6|2483.6200103759766|
|     997124|             5|1339.3400039672852|
|     996985|             5|1365.0799713134766|
|     996959|             5|1536.2200164794922|
|     996939|             5| 2344.730010986328|
|     996492|             5|1443.2499694824219|
|     995857|             5|1336.5499877929688|
|     994152|             5| 1239.109992980957|
|     993982|             5|1494.3799896240234|
|     993362|             5|1571.5800018

In [39]:
df= df_orders.groupBy('customer_id',year('order_date').alias('year'))\
           .agg(sum(col('price')*col('quantity')).alias('yearly_spending'))
df.show()

df_final = df.groupBy('customer_id')\
                    .agg(
                        countDistinct('year').alias('distinct_order_years'),
                        sum('yearly_spending').alias('total_spending'))\
                    .filter(col('distinct_order_years')>=5)\
                    .sort(col('customer_id').desc(),col('distinct_order_years').desc())

df_final.show()

+-----------+----+------------------+
|customer_id|year|   yearly_spending|
+-----------+----+------------------+
|     388926|2000|127.56999969482422|
|      73005|2016| 153.1199951171875|
|     785194|2012|  256.739990234375|
|     805598|2017| 513.5399780273438|
|     291673|2016|            344.25|
|     171887|2005|160.08999633789062|
|     602684|2017|199.58999633789062|
|     102973|2019| 324.3599853515625|
|     240379|2012|             179.0|
|     553700|2009| 282.4599914550781|
|     442332|2020| 555.9000244140625|
|     681505|2006|111.55000305175781|
|      97737|2005| 241.9199981689453|
|     750648|2002|187.25999450683594|
|     523868|2006| 398.7799987792969|
|     310804|2012|456.77996826171875|
|     480498|2002| 513.2099914550781|
|     526920|2006| 525.8399658203125|
|      68965|2001|122.69999694824219|
|     269098|2013|420.05999755859375|
+-----------+----+------------------+
only showing top 20 rows



[Stage 160:=================================>                   (125 + 4) / 200]

+-----------+--------------------+------------------+
|customer_id|distinct_order_years|    total_spending|
+-----------+--------------------+------------------+
|     999747|                   5|1927.0099792480469|
|     999502|                   6|2031.1599884033203|
|     998877|                   5|1861.1299285888672|
|     998725|                   6| 2014.740005493164|
|     998362|                   5|1849.7599792480469|
|     998347|                   5| 1494.969985961914|
|     997974|                   5|1628.3100357055664|
|     997930|                   5|1911.6500091552734|
|     997612|                   6|2483.6200103759766|
|     997124|                   5|1339.3400039672852|
|     996985|                   5|1365.0799713134766|
|     996959|                   5|1536.2200164794922|
|     996939|                   5| 2344.730010986328|
|     996492|                   5|1443.2499694824219|
|     995857|                   5|1336.5499877929688|
|     994152|               

In [40]:
df_orders.groupBy('customer_id',year('order_date').alias('year'))\
           .agg(sum(col('price')*col('quantity')).alias('yearly_spending'))\
           .groupBy('customer_id')\
                    .agg(countDistinct('year').alias('distinct_order_years'),sum('yearly_spending').alias('total_spending'))\
                    .filter(col('distinct_order_years')>=5)\
                    .sort(col('customer_id').desc(),col('distinct_order_years').desc())\
                    .show()

[Stage 163:==========================================>          (160 + 4) / 200]

+-----------+--------------------+------------------+
|customer_id|distinct_order_years|    total_spending|
+-----------+--------------------+------------------+
|     999747|                   5|1927.0099792480469|
|     999502|                   6|2031.1599884033203|
|     998877|                   5|1861.1299285888672|
|     998725|                   6| 2014.740005493164|
|     998362|                   5|1849.7599792480469|
|     998347|                   5| 1494.969985961914|
|     997974|                   5|1628.3100357055664|
|     997930|                   5|1911.6500091552734|
|     997612|                   6|2483.6200103759766|
|     997124|                   5|1339.3400039672852|
|     996985|                   5|1365.0799713134766|
|     996959|                   5|1536.2200164794922|
|     996939|                   5| 2344.730010986328|
|     996492|                   5|1443.2499694824219|
|     995857|                   5|1336.5499877929688|
|     994152|               

In [58]:
#spark.sql

df_orders.show(3)

spark.sql('''
            with cte_1
            as
                (
                    select customer_id,year(order_date) as order_year,sum(price*quantity) as total_amount_year
                    from sql_orders
                    group by customer_id,year(order_date)
                )
            select customer_id,count(order_year) as distinct_years,sum(total_amount_year) as total_amount
            from cte_1
            group by customer_id
            having  count(order_year) >=5
            order by customer_id desc,distinct_years desc
           
    
        ''').show(5)

+--------+-----------+----------+------+----------+------------+-------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|  state|quantity|
+--------+-----------+----------+------+----------+------------+-------+--------+
|       1|     192520|        70|152.33|2007-10-10|      PLACED| Odisha|       2|
|       2|     835421|        34|117.42|2008-12-02|     SHIPPED| Kerala|       3|
|       3|     159165|        13|128.85|2010-04-22|   DELIVERED|Gujarat|       3|
+--------+-----------+----------+------+----------+------------+-------+--------+
only showing top 3 rows



+-----------+--------------+------------------+
|customer_id|distinct_years|      total_amount|
+-----------+--------------+------------------+
|     999747|             5|1927.0099792480469|
|     999502|             6|2031.1599884033203|
|     998877|             5|1861.1299285888672|
|     998725|             6| 2014.740005493164|
|     998362|             5|1849.7599792480469|
+-----------+--------------+------------------+
only showing top 5 rows



# Question 5

5. Consider only orders placed after 2005, and for each `order_status` calculate the average order value (`unit_price * quantity`), sorting the result in descending order of average value.

In [68]:
#pratice
#ankit
df_orders.filter(year(col('order_date'))>2005)\
              .withColumn("order_value", col("price") * col("quantity")) \
              .groupBy("order_status") \
              .agg(avg("order_value").alias("avg_order_value")) \
              .orderBy(col("avg_order_value").desc()).show()


[Stage 219:==============>                                          (1 + 3) / 4]

+------------+------------------+
|order_status|   avg_order_value|
+------------+------------------+
|     SHIPPED|300.32913405599123|
|   CANCELLED|300.20234940687783|
|    RETURNED| 299.9358956805303|
|   DELIVERED| 299.8343608209149|
|      PLACED|299.38588297247287|
+------------+------------------+



In [67]:
df_orders.where(year('order_date')>2005)\
            .withColumn('order_value',col('price')*col('quantity'))\
            .groupBy('order_status')\
                .agg(
                    avg("order_value").alias("avg_order_value")
                    )\
            .orderBy(col("avg_order_value").desc())\
            .show()

[Stage 217:==========================================>              (3 + 1) / 4]

+------------+------------------+
|order_status|   avg_order_value|
+------------+------------------+
|     SHIPPED|300.32913405599123|
|   CANCELLED|300.20234940687783|
|    RETURNED| 299.9358956805303|
|   DELIVERED| 299.8343608209149|
|      PLACED|299.38588297247287|
+------------+------------------+



In [69]:
#spark.sql

spark.sql('''
            select order_status,avg(price*quantity) as avg_order_value
            from sql_orders
            where year(order_date)>2005
            group by order_status
            order by avg(price*quantity) desc

''').show()

[Stage 221:==============>                                          (1 + 3) / 4]

+------------+------------------+
|order_status|   avg_order_value|
+------------+------------------+
|     SHIPPED|300.32913405599123|
|   CANCELLED|300.20234940687783|
|    RETURNED| 299.9358956805303|
|   DELIVERED| 299.8343608209149|
|      PLACED|299.38588297247287|
+------------+------------------+



# Question 6

6. Normalize the `state` column by trimming spaces and converting it to lowercase, 
then group by the normalized state and compute the total quantity sold, 
displaying only the top 5 states by total quantity.

In [28]:
df_orders.withColumn('state',trim(col('state')))\
            .withColumn('state',lower(col('state')))\
            .groupBy(col('state'))\
                .agg(sum(col('quantity')).alias('total_quantity_per_state'))\
                .orderBy(col('total_quantity_per_state').desc())\
                .limit(5)\
                .show()

#----------OR---not required here--bit complex

window_spec = Window.partitionBy(col('state'))
window_spec2 = Window.orderBy(col('total_quantity_per_state').desc())

df_orders.withColumn('state',trim(col('state')))\
            .withColumn('state',lower(col('state')))\
            .withColumn('total_quantity_per_state',sum(col('quantity')).over(window_spec))\
            .select('state','total_quantity_per_state')\
            .dropDuplicates()\
            .withColumn('dense_rank',dense_rank().over(window_spec2))\
            .show()

26/04/25 11:26:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------------+------------------------+
|           state|total_quantity_per_state|
+----------------+------------------------+
|          kerala|                   91678|
|himachal pradesh|                   91537|
|          odisha|                   91485|
|           bihar|                   91474|
|  andhra pradesh|                   91258|
+----------------+------------------------+



[Stage 96:====================================>                 (137 + 4) / 200]

+----------------+------------------------+----------+
|           state|total_quantity_per_state|dense_rank|
+----------------+------------------------+----------+
|          kerala|                   91678|         1|
|himachal pradesh|                   91537|         2|
|          odisha|                   91485|         3|
|           bihar|                   91474|         4|
|  andhra pradesh|                   91258|         5|
|      tamil nadu|                   91198|         6|
|     maharashtra|                   91162|         7|
|           assam|                   91072|         8|
|         haryana|                   91069|         9|
|    chhattisgarh|                   91065|        10|
|       jharkhand|                   91001|        11|
|       karnataka|                   90926|        12|
|           delhi|                   90825|        13|
|     uttarakhand|                   90787|        14|
|         gujarat|                   90580|        15|
|   uttar 

In [24]:
#spark.sql

spark.sql('''
            with cte_1 as
                (
                    select state,sum(quantity) as total_quantity_state
                    from sql_orders
                    group by state
                ),
            cte_2 as
                (
                    select *,dense_rank() over(order by total_quantity_state desc) as dense_rank
                    from cte_1
                )
            select * from cte_2 where dense_rank<=5         
                
''').show()

26/04/25 11:20:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 66:======================================>               (144 + 4) / 200]

+----------------+--------------------+----------+
|           state|total_quantity_state|dense_rank|
+----------------+--------------------+----------+
|          Kerala|               91678|         1|
|Himachal Pradesh|               91537|         2|
|          Odisha|               91485|         3|
|           Bihar|               91474|         4|
|  Andhra Pradesh|               91258|         5|
+----------------+--------------------+----------+



# Question 7

7. For each year, calculate the total revenue generated, and return only those years where the average unit price for that year is greater than the overall average unit price across all orders.

In [40]:
#avg price acroos all---(price*quantity)/total orders

avg_price_overal = df_orders.agg(avg(col('price')).alias('avg_price_overal')).collect()[0][0]


df_orders.withColumn("order_year", year(to_date("order_date"))) \
        .groupBy(col('order_year'))\
            .agg(
                    sum(col('price')*col('quantity')).alias('total_revenue_year'),
                    avg(col('price')).alias('avg_price_year')
                )\
            .filter(col('avg_price_year') > avg_price_overal)\
            .show()

+----------+--------------------+------------------+
|order_year|  total_revenue_year|    avg_price_year|
+----------+--------------------+------------------+
|      2003|1.1993534145317078E7|150.13609023236455|
|      2007|1.1783352760177612E7| 150.1715764254729|
|      2018|1.2048305569709778E7|150.01082629638478|
|      2015|1.1978013062454224E7|150.19526505585708|
|      2023|1.1935240140686035E7| 150.1126612599448|
|      2013|1.1890355756332397E7| 150.0150303117444|
|      2014|1.1867562521339417E7| 150.1036789193327|
|      2019|1.1991442129745483E7| 150.0913369110557|
|      2004|1.2163106137672424E7| 150.0990046523083|
|      2012| 1.198793772052002E7| 150.0411894012096|
|      2016|1.2093026498901367E7|150.06118171630186|
|      2005|1.2083784159446716E7|150.26019669518237|
|      2000|1.2038766014564514E7|150.03285701208492|
|      2017| 1.191785516029358E7|150.09097871080326|
+----------+--------------------+------------------+



In [44]:
#sql

spark.sql('''
            with cte_1 as 
                (
                select avg(price) as avg_price from sql_orders
                ),
            cte_2 as 
                (
                    select year(order_date),
                    sum(price*quantity) as total_revenue_year,
                    avg(price) as avg_price_year
                    from sql_orders
                    group by year(order_date)
                )
            select cte_2.* 
            from cte_2 
            join
            cte_1
            on  cte_2.avg_price_year>cte_1.avg_price
            

''').show()

+----------------+--------------------+------------------+
|year(order_date)|  total_revenue_year|    avg_price_year|
+----------------+--------------------+------------------+
|            2003|1.1993534145317078E7|150.13609023236455|
|            2007|1.1783352760177612E7| 150.1715764254729|
|            2018|1.2048305569709778E7|150.01082629638478|
|            2015|1.1978013062454224E7|150.19526505585708|
|            2023|1.1935240140686035E7| 150.1126612599448|
|            2013|1.1890355756332397E7| 150.0150303117444|
|            2014|1.1867562521339417E7| 150.1036789193327|
|            2019|1.1991442129745483E7| 150.0913369110557|
|            2004|1.2163106137672424E7| 150.0990046523083|
|            2012| 1.198793772052002E7| 150.0411894012096|
|            2016|1.2093026498901367E7|150.06118171630186|
|            2005|1.2083784159446716E7|150.26019669518237|
|            2000|1.2038766014564514E7|150.03285701208492|
|            2017| 1.191785516029358E7|150.0909787108032

8. Create a column bulk_order_flag such that quantity greater than or equal to 3 is marked as Y and others as N, then for each state calculate the percentage of bulk orders.

In [48]:
#pratice
df_orders.withColumn('bulk_order_flag',when(col('quantity')>=3,'Y').otherwise('N'))\
        .groupBy(col('state'))\
            .agg(
                count('*').alias('no_of_orders_statewise'),
                sum(when(col('bulk_order_flag')=='Y',1).otherwise(0)).alias('no_of_bulk_order_statewise')
            )\
        .withColumn('%bulk_order_statewise',round((col('no_of_bulk_order_statewise')/col('no_of_orders_statewise'))*100,2))\
        .show()

+----------------+----------------------+--------------------------+---------------------+
|           state|no_of_orders_statewise|no_of_bulk_order_statewise|%bulk_order_statewise|
+----------------+----------------------+--------------------------+---------------------+
|       Karnataka|                 45555|                     15091|                33.13|
|          Odisha|                 45719|                     15201|                33.25|
|          Kerala|                 45691|                     15362|                33.62|
|      Tamil Nadu|                 45638|                     15257|                33.43|
|    Chhattisgarh|                 45562|                     15101|                33.14|
|  Andhra Pradesh|                 45650|                     15179|                33.25|
|  Madhya Pradesh|                 45254|                     15036|                33.23|
|          Punjab|                 45479|                     14874|                32.71|

In [60]:
#spark.sql

spark.sql('''
            with cte_1 as 
                (
                  select * ,
                      CASE
                          when quantity>=3 then 'Y'
                          else 'N'
                        END as bulk_order_flag
                    from sql_orders
                        
                ),
            cte_2 as 
                (
                    select state,
                            count(*) as total_order_statewise,
                            sum(
                                CASE
                                    when bulk_order_flag=='Y' then 1
                                    else 0
                                END
                                ) as no_of_bulk_order_statewise
                            from cte_1
                            group by state
                )
            select *,(no_of_bulk_order_statewise/total_order_statewise)*100 as pct_bulk_order from cte_2
            

''').show()


+----------------+---------------------+--------------------------+------------------+
|           state|total_order_statewise|no_of_bulk_order_statewise|    pct_bulk_order|
+----------------+---------------------+--------------------------+------------------+
|       Karnataka|                45555|                     15091|  33.1269893535287|
|          Odisha|                45719|                     15201| 33.24875872175682|
|          Kerala|                45691|                     15362| 33.62150095204745|
|      Tamil Nadu|                45638|                     15257|33.430474604496254|
|    Chhattisgarh|                45562|                     15101| 33.14384794346166|
|  Andhra Pradesh|                45650|                     15179| 33.25082146768894|
|  Madhya Pradesh|                45254|                     15036| 33.22579219516506|
|          Punjab|                45479|                     14874| 32.70520459992524|
|             Goa|                45090|   

# Question 9

9. Find all states where the difference between the maximum and minimum unit price within that state is less than 100, and return the state along with the price difference.

In [63]:
df_orders.groupBy('state')\
            .agg(
                max('price').alias('max_price'),
                min('price').alias('min_price')
                )\
            .withColumn('price_diff',col('max_price')-col('min_price'))\
            .filter(col('price_diff')<100)\
                .show()

# df = df_orders.groupBy("state") \
#               .agg(
#                   (max("unit_price") - min("unit_price")).alias("price_difference")
#               ) \
#               .filter(col("price_difference") < 100)

+----------------+----------+
|           state|price_diff|
+----------------+----------+
|  Madhya Pradesh|     99.98|
|Himachal Pradesh| 99.990005|
+----------------+----------+



In [62]:
#spark.sql
spark.sql('''
            select * ,(max_price - min_price) as diff_price
            from 
                (
                    select state,max(price) as max_price,min(price) as min_price
                    from sql_orders
                    group by state
                )
            where max_price - min_price<100 
''').show()

+----------------+---------+---------+----------+
|           state|max_price|min_price|diff_price|
+----------------+---------+---------+----------+
|  Madhya Pradesh|   199.99|   100.01|     99.98|
|Himachal Pradesh|   199.99|    100.0| 99.990005|
+----------------+---------+---------+----------+



# Question 10

10. find all states where orders have been placed in at least three different years and the average order value is greater than the overall average order value across all orders. Consider only orders placed after 2005 and exclude cancelled orders. For the qualifying states, return the state name, total number of orders, total revenue, and average order value, and sort the result by total revenue in descending order.

In [80]:
#pratice
#step-1 orders placed after 2005,order_status not cancelled
#step-2 save the overall avg order value
#step-3 groupby state

df_base = df_orders.filter(year(col('order_date'))>2005)\
                    .filter(lower(col('order_status'))!='cancelled')

overal_avg_order_value = df_base.agg(avg((col('price')*col('quantity')))).collect()[0][0]
print(overal_avg_order_value)

df_result = df_base.withColumn('order_year',year(col('order_date')))\
                    .groupBy(col('state'))\
                        .agg(
                            count('*').alias('total_no_orders'),
                            sum(col('price')*col('quantity')).alias('total_revenue'),
                            avg(col('price')*col('quantity')).alias('avg_order_value'),
                            countDistinct(col('order_year')).alias('no_of_distinct_years')
                            
                        )\
                    .filter(col('avg_order_value')>overal_avg_order_value)\
                    .filter(col('no_of_distinct_years')>=3)\
                    .orderBy(col('total_revenue').desc())\
                    .show()

299.8099221113456


[Stage 353:==========================================>          (162 + 4) / 200]

+----------------+---------------+-----------------+------------------+--------------------+
|           state|total_no_orders|    total_revenue|   avg_order_value|no_of_distinct_years|
+----------------+---------------+-----------------+------------------+--------------------+
|           Bihar|          33070|9920420.409629822| 299.9824738321688|                  19|
|Himachal Pradesh|          33026|9918818.682037354| 300.3336365904849|                  19|
|          Kerala|          32940|9914585.815734863|  300.989247593651|                  19|
|     Maharashtra|          32895| 9889949.21144104| 300.6520508114011|                  19|
|          Odisha|          32930|9886126.528282166|  300.216414463473|                  19|
|    Chhattisgarh|          32888| 9862969.16571045|299.89568127312236|                  19|
|     Uttarakhand|          32817|9841315.718765259| 299.8846853388566|                  19|
|       Rajasthan|          32506|9771719.679206848| 300.6128000740432

In [86]:
#ankit


overall_avg = (
    df_orders
    .withColumn("order_date", to_date("order_date"))
    .filter((col("order_status") != "CANCELLED") & (year("order_date") > 2005))
    .withColumn("order_value", col("price") * col("quantity"))
    .agg(avg("order_value").alias("avg_order_value"))
    .collect()[0][0]
)
print(overall_avg)
# final solution
result_df = (
    df_orders
    .withColumn("order_date", to_date("order_date"))
    .filter((col("order_status") != "CANCELLED") & (year("order_date") > 2005))
    .withColumn("order_year", year("order_date"))
    .withColumn("order_value", col("price") * col("quantity"))
    .groupBy("state")
    .agg(
        count("*").alias("total_orders"),
        sum("order_value").alias("total_revenue"),
        avg("order_value").alias("avg_order_value"),
        countDistinct("order_year").alias("active_years")
    )
    .filter(
        (col("active_years") >= 3) &
        (col("avg_order_value") > overall_avg)
    )
    .orderBy(col("total_revenue").desc())
)

result_df.show()

299.8099221113456


[Stage 390:======================================>              (146 + 4) / 200]

+----------------+------------+-----------------+------------------+------------+
|           state|total_orders|    total_revenue|   avg_order_value|active_years|
+----------------+------------+-----------------+------------------+------------+
|           Bihar|       33070|9920420.409629822| 299.9824738321688|          19|
|Himachal Pradesh|       33026|9918818.682037354| 300.3336365904849|          19|
|          Kerala|       32940|9914585.815734863|  300.989247593651|          19|
|     Maharashtra|       32895| 9889949.21144104| 300.6520508114011|          19|
|          Odisha|       32930|9886126.528282166|  300.216414463473|          19|
|    Chhattisgarh|       32888| 9862969.16571045|299.89568127312236|          19|
|     Uttarakhand|       32817|9841315.718765259| 299.8846853388566|          19|
|       Rajasthan|       32506|9771719.679206848| 300.6128000740432|          19|
|             Goa|       32543| 9768392.94078064| 300.1687902400098|          19|
|     West Benga

In [ ]:
spark.stop()